# Topic Modeling with NMF

## Setup and Imports

In [20]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation as LDA, NMF 

from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA, TruncatedSVD as SVD

In [2]:
sns.set_theme(style="white")
colors = "YlGnBu"

In [3]:
model_type = 'nmf' # or 'nmf'
data_home = "../input"


In [4]:
import os

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

In [5]:
OHCO = ['doc_title', 'para_num', 'sentence_num', 'token_num']
SENTS = OHCO[:3]
PARAS = OHCO[:2]
STORIES = OHCO[:1]

BAG = PARAS

In [6]:
BAG

['doc_title', 'para_num']

In [7]:
TOKENS = pd.read_csv('data/p2591-TOKENS.csv').set_index(OHCO).dropna()
TOKENS

pos_tuple  pos token_str  \
doc_title para_num sentence_num token_num                                   
ASHPUTTEL 0        0            0            ('The', 'DT')   DT       The   
                                1           ('wife', 'NN')   NN      wife   
                                2             ('of', 'IN')   IN        of   
                                3              ('a', 'DT')   DT         a   
                                4           ('rich', 'JJ')   JJ      rich   
...                                                    ...  ...       ...   
TOM THUMB 21       3            40            ('s', 'VBZ')  VBZ         s   
                                41            ('no', 'DT')   DT        no   
                                42         ('place', 'NN')   NN     place   
                                43          ('like', 'IN')   IN      like   
                                44          ('HOME', 'NN')   NN      HOME   

                                          term_str pos_group  
doc_title para_num sentence_num token_num                     
ASHPUTTEL 0        0            0              the        DT  
                                1             wife        NN  
                                2               of        IN  
                                3                a        DT  
                                4             rich        JJ  
...                                            ...       ...  
TOM THUMB 21       3            40               s        VB  
                                41              no        DT  
                                42           place        NN  
                                43            like        IN  
                                44            home        NN  

[101046 rows x 5 columns]

In [8]:
DOCS = TOKENS[TOKENS.pos.str.match(r'^NNS?$')]\
    .groupby(BAG).term_str\
    .apply(lambda x: ' '.join(map(str,x)))\
    .to_frame()\
    .rename(columns={'term_str':'doc_str'})

DOCS

doc_str
doc_title para_num                                                   
ASHPUTTEL 0         wife man end drew daughter girl watch afterwar...
          1         work daylight water fire sisters sorts ways ev...
          2         father fair wife s daughters clothes diamonds ...
          3         king land feast days son bride sisters hair sh...
          4                             peas ashes maiden door garden
...                                                               ...
TOM THUMB 17        wolf night house drain kitchen pantry ate dran...
          18        shout noise wolf everybody house clatter man mind
          19        woodman wife noise crack door wolf woodman axe...
          20                                             riches world
          21        son plenty clothes ones journey home father mo...

[889 rows x 1 columns]

## Create Vector Space

In [9]:
from sklearn.feature_extraction import text

my_stop_words = list(text.ENGLISH_STOP_WORDS.union(['yes']))
my_stop_words[:10]

['whoever',
 'system',
 'another',
 'ten',
 'each',
 'but',
 'something',
 'all',
 'yourselves',
 'becoming']

In [10]:
# count_engine = CountVectorizer(max_df=.9, min_df=2, stop_words=my_stop_words) # Got some advice from clause to lower min ax max df because corpus ins amll
# count_model = count_engine.fit_transform(DOCS.doc_str)
# TERMS = count_engine.get_feature_names_out()
# VOCAB = pd.DataFrame(index=TERMS)
# VOCAB.index.name = 'term_str'
# DTM = pd.DataFrame(count_model.toarray(), index=DOCS.index, columns=TERMS)
# DTM

In [11]:
# Used claude code to help with tfidf engine and model because I want nmf to get better topics than lda
tfidf_engine = TfidfVectorizer(max_df=.75, min_df=5, stop_words=my_stop_words)
tfidf_model = tfidf_engine.fit_transform(DOCS.doc_str)
TERMS = tfidf_engine.get_feature_names_out()
TFIDF = tfidf_engine.fit_transform(DOCS.doc_str)
VOCAB = pd.DataFrame(index=TERMS)
VOCAB.index.name = 'term_str'
DTM = pd.DataFrame(tfidf_model.toarray(), index=DOCS.index, columns=TERMS)
DTM


account  advice      air  alas  ale  anger  animals  \
doc_title para_num                                                        
ASHPUTTEL 0             0.0     0.0  0.00000   0.0  0.0    0.0      0.0   
          1             0.0     0.0  0.00000   0.0  0.0    0.0      0.0   
          2             0.0     0.0  0.00000   0.0  0.0    0.0      0.0   
          3             0.0     0.0  0.00000   0.0  0.0    0.0      0.0   
          4             0.0     0.0  0.00000   0.0  0.0    0.0      0.0   
...                     ...     ...      ...   ...  ...    ...      ...   
TOM THUMB 17            0.0     0.0  0.00000   0.0  0.0    0.0      0.0   
          18            0.0     0.0  0.00000   0.0  0.0    0.0      0.0   
          19            0.0     0.0  0.12911   0.0  0.0    0.0      0.0   
          20            0.0     0.0  0.00000   0.0  0.0    0.0      0.0   
          21            0.0     0.0  0.00000   0.0  0.0    0.0      0.0   

                    answer  apple  apples  ...  woods  word  words      work  \
doc_title para_num                         ...                                 
ASHPUTTEL 0            0.0    0.0     0.0  ...    0.0   0.0    0.0  0.000000   
          1            0.0    0.0     0.0  ...    0.0   0.0    0.0  0.280609   
          2            0.0    0.0     0.0  ...    0.0   0.0    0.0  0.000000   
          3            0.0    0.0     0.0  ...    0.0   0.0    0.0  0.000000   
          4            0.0    0.0     0.0  ...    0.0   0.0    0.0  0.000000   
...                    ...    ...     ...  ...    ...   ...    ...       ...   
TOM THUMB 17           0.0    0.0     0.0  ...    0.0   0.0    0.0  0.000000   
          18           0.0    0.0     0.0  ...    0.0   0.0    0.0  0.000000   
          19           0.0    0.0     0.0  ...    0.0   0.0    0.0  0.000000   
          20           0.0    0.0     0.0  ...    0.0   0.0    0.0  0.000000   
          21           0.0    0.0     0.0  ...    0.0   0.0    0.0  0.000000   

                       world  wretch   ye  year  years  youth  
doc_title para_num                                             
ASHPUTTEL 0         0.000000     0.0  0.0   0.0    0.0    0.0  
          1         0.000000     0.0  0.0   0.0    0.0    0.0  
          2         0.000000     0.0  0.0   0.0    0.0    0.0  
          3         0.000000     0.0  0.0   0.0    0.0    0.0  
          4         0.000000     0.0  0.0   0.0    0.0    0.0  
...                      ...     ...  ...   ...    ...    ...  
TOM THUMB 17        0.000000     0.0  0.0   0.0    0.0    0.0  
          18        0.000000     0.0  0.0   0.0    0.0    0.0  
          19        0.105347     0.0  0.0   0.0    0.0    0.0  
          20        1.000000     0.0  0.0   0.0    0.0    0.0  
          21        0.000000     0.0  0.0   0.0    0.0    0.0  

[889 rows x 573 columns]

## Generate Model with 20 Topics

In [12]:
n_topics = 5
max_iter = 100
n_top_terms = 5
TNAMES = [f"T{str(x).zfill(len(str(n_topics)))}" for x in range(n_topics)]

In [13]:
if model_type == 'lda':
    topic_engine = LDA(n_components=n_topics, max_iter=max_iter)
elif model_type == 'nmf':
    topic_engine = NMF(n_components=n_topics, max_iter=max_iter)
topic_model = topic_engine.fit_transform(tfidf_model)

## THETA

In [14]:
THETA = pd.DataFrame(topic_model, index=DOCS.index, columns=TNAMES)
THETA.columns.name = 'topic_id'
THETA.sample(10).T.style.background_gradient(cmap=colors, axis=None)

doc_title,THE JUNIPER-TREE,JORINDA AND JORINDEL,THE GOLDEN GOOSE,"THE MOUSE, THE BIRD, AND THE SAUSAGE",THE GOLDEN GOOSE,ASHPUTTEL,THE GOLDEN BIRD,THE JUNIPER-TREE,CLEVER HANS,LILY AND THE LION
para_num,17,13,13,4,20,26,16,3,1,3
topic_id,,,,,,,,,,
T0,0.014702,0.051292,0.042137,0.011495,0.152881,0.032966,0.046072,0.078430,0.005393,0.034735
T1,0.095997,0.000000,0.006820,0.019767,0.000000,0.000000,0.184543,0.000000,0.000000,0.000000
T2,0.008205,0.000000,0.000000,0.018908,0.000000,0.005609,0.007198,0.003274,0.000000,0.143582
T3,0.008365,0.029965,0.007455,0.089328,0.000000,0.000000,0.003732,0.027291,0.017460,0.052987
T4,0.023982,0.028494,0.008528,0.000000,0.000000,0.000000,0.000000,0.075374,0.243948,0.000000


## PHI

In [15]:
PHI = pd.DataFrame(topic_engine.components_, columns=TERMS, index=TNAMES)
PHI.index.name = 'topic_id'
PHI.columns.name = 'term_str'
PHI.T.sample(10).T.style.background_gradient(cmap=colors, axis=None)

term_str,room,clothes,sheep,boat,parlour,evening,fellow,stone,shame,draught
topic_id,,,,,,,,,,
T0,0.097058,0.104188,0.008280,0.041348,0.006922,0.131661,0.043357,0.081316,0.011024,0.054295
T1,0.001798,0.007069,0.000000,0.000000,0.001435,0.000000,0.000000,0.057167,0.000000,0.000000
T2,0.000000,0.019895,0.000000,0.033732,0.011769,0.000000,0.024370,0.000000,0.000685,0.006130
T3,0.126288,0.022116,0.030734,0.000000,0.023551,0.044373,0.016401,0.013081,0.000692,0.013858
T4,0.000000,0.014936,0.000000,0.000000,0.000000,0.150708,0.000000,0.005759,0.001597,0.000000


## Get Top Terms By Topic

In [16]:
TOPICS = PHI.stack().groupby('topic_id')\
    .apply(lambda x: ' '.join(x.sort_values(ascending=False).head(n_top_terms).reset_index().term_str))\
    .to_frame('top_terms')
TOPICS


,top_terms
topic_id,
T0,king man wife daughter day
T1,bird chain fox man gold
T2,father son home brother bread
T3,wolf door house woman cat
T4,mother hans day son alas


## PCA + LDA

In [29]:
THETA

topic_id                  T0        T1        T2        T3        T4
doc_title para_num                                                  
ASHPUTTEL 0         0.099067  0.000000  0.067677  0.036881  0.011113
          1         0.018599  0.000000  0.001211  0.049725  0.026338
          2         0.056299  0.066726  0.070309  0.010374  0.080729
          3         0.102598  0.000000  0.013530  0.000000  0.083139
          4         0.007571  0.000000  0.000000  0.109821  0.000237
...                      ...       ...       ...       ...       ...
TOM THUMB 17        0.005702  0.000744  0.010610  0.211304  0.000000
          18        0.011705  0.013792  0.000000  0.183094  0.000000
          19        0.002799  0.000000  0.238230  0.202034  0.000000
          20        0.020416  0.000000  0.014306  0.007843  0.000000
          21        0.022306  0.000000  0.132834  0.017031  0.114274

[889 rows x 5 columns]

In [22]:
pca_engine = PCA(n_components=5)
DCM_THETA = pd.DataFrame(pca_engine.fit_transform(THETA), index=THETA.index)
DCM_THETA.columns = ['PC{}'.format(i) for i in DCM_THETA.columns]
DCM_THETA

PC0       PC1       PC2       PC3       PC4
doc_title para_num                                                  
ASHPUTTEL 0        -0.008545 -0.022830 -0.043840  0.034268  0.050534
          1         0.024362 -0.010294  0.005123 -0.014660 -0.021979
          2        -0.087110 -0.001497  0.015483  0.001392  0.025842
          3        -0.037595 -0.030018 -0.035034 -0.058029  0.050030
          4         0.072543 -0.007042  0.037620  0.017092 -0.017546
...                      ...       ...       ...       ...       ...
TOM THUMB 17        0.126202 -0.025192  0.106544  0.055050  0.014320
          18        0.109684 -0.006518  0.090814  0.039581  0.011801
          19        0.006686 -0.110753  0.096493  0.228234  0.028706
          20        0.006078  0.003060 -0.036401 -0.002393 -0.037315
          21       -0.103290 -0.095759  0.020354  0.028906 -0.001705

[889 rows x 5 columns]

## Save Files to Output

In [17]:
THETA.to_csv(f"{output_dir}/pg2591-THETA.csv", index=True)
PHI.to_csv(f"{output_dir}/pg2591-PHI.csv", index=True)
TOPICS.to_csv(f"{output_dir}/pg2591-TOPICS.csv", index=True)